# Baseline2 Pipeline

Notebook này chạy từng script của `baseline2` theo đúng thứ tự pipeline. Mỗi cell sẽ stream log trực tiếp từ script tương ứng.


In [2]:
from pathlib import Path
import os
import subprocess
import sys

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'GRACE-improve').exists():
            return candidate
    raise RuntimeError('Could not locate repo root containing GRACE-improve')

REPO_ROOT = find_repo_root()
BASELINE2_DIR = REPO_ROOT / 'GRACE-improve' / 'baseline' / 'baseline2'
PYTHON = sys.executable

for name in [
    'GRACE_FEATURE_LIMIT',
    'GRACE_FEATURE_LIMIT_TRAIN',
    'GRACE_FEATURE_LIMIT_VAL',
    'GRACE_FEATURE_LIMIT_TEST',
    'GRACE_FORCE_REBUILD_FEATURES',
    'GRACE_MAX_TEST_SAMPLES',
]:
    os.environ.pop(name, None)

os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['GRACE_DATASET'] = 'devign'
os.environ['GRACE_PREFILTER_MODEL_NAME'] = 'hybrid_multiview_prefilter'
os.environ['GRACE_RETRIEVAL_MODEL_ID'] = 'microsoft/unixcoder-base-nine'
os.environ['GRACE_GRAPH_BACKEND'] = 'auto'
os.environ['GRACE_AUTO_DOWNLOAD_MISSING'] = '1'
os.environ['GRACE_LOAD_IN_4BIT'] = '1'
os.environ['GRACE_CALL_LLM_FOR_INSPECT'] = '1'
os.environ['GRACE_CALL_LLM_FOR_HIGH'] = '1'

def run_script(script_name: str, extra_env: dict | None = None) -> None:
    env = os.environ.copy()
    if extra_env:
        env.update({key: str(value) for key, value in extra_env.items()})
    command = [PYTHON, str(BASELINE2_DIR / script_name)]
    print(f'Running: {command[0]} {command[1]}')
    process = subprocess.Popen(
        command,
        cwd=str(REPO_ROOT),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f'{script_name} failed with exit code {return_code}')

print('Python:', PYTHON)
print('Repo root:', REPO_ROOT)
print('Baseline2 dir:', BASELINE2_DIR)


Python: c:\Users\Admin\Documents\miniconda\envs\nckh\python.exe
Repo root: C:\Users\Admin\Documents\1. UET\lab\VulGuardVN
Baseline2 dir: C:\Users\Admin\Documents\1. UET\lab\VulGuardVN\GRACE-improve\baseline\baseline2


In [2]:
run_script('00_verify_assets.py')


Running: c:\Users\Admin\Documents\miniconda\envs\nckh\python.exe C:\Users\Admin\Documents\1. UET\lab\VulGuardVN\GRACE-improve\baseline\baseline2\00_verify_assets.py
{
  "dataset": "devign",
  "auto_download_missing": true,
  "actions": [],
  "splits": {
    "split_dir": "C:\\Users\\Admin\\Documents\\1. UET\\lab\\VulGuardVN\\GRACE-improve\\baseline\\artifacts\\splits\\devign",
    "available": true,
    "files": {
      "train": "C:\\Users\\Admin\\Documents\\1. UET\\lab\\VulGuardVN\\GRACE-improve\\baseline\\artifacts\\splits\\devign\\train.jsonl",
      "val": "C:\\Users\\Admin\\Documents\\1. UET\\lab\\VulGuardVN\\GRACE-improve\\baseline\\artifacts\\splits\\devign\\val.jsonl",
      "test": "C:\\Users\\Admin\\Documents\\1. UET\\lab\\VulGuardVN\\GRACE-improve\\baseline\\artifacts\\splits\\devign\\test.jsonl"
    }
  },
  "semantic_model": {
    "repo_id": "microsoft/unixcoder-base-nine",
    "path": "C:\\Users\\Admin\\Documents\\1. UET\\lab\\VulGuardVN\\GRACE-improve\\baseline\\artifacts

In [3]:
run_script('01_prepare_datasets.py')


Running: c:\Users\Admin\Documents\miniconda\envs\nckh\python.exe C:\Users\Admin\Documents\1. UET\lab\VulGuardVN\GRACE-improve\baseline\baseline2\01_prepare_datasets.py
devign: 27318 rows indexed at C:\Users\Admin\Documents\1. UET\lab\VulGuardVN\GRACE-improve\baseline\artifacts\processed\devign\index.csv


In [4]:
run_script('02_create_splits.py')


Running: c:\Users\Admin\Documents\miniconda\envs\nckh\python.exe C:\Users\Admin\Documents\1. UET\lab\VulGuardVN\GRACE-improve\baseline\baseline2\02_create_splits.py
devign: split files written to C:\Users\Admin\Documents\1. UET\lab\VulGuardVN\GRACE-improve\baseline\artifacts\splits\devign


In [5]:
run_script('03_build_feature_store.py')


Running: c:\Users\Admin\Documents\miniconda\envs\nckh\python.exe C:\Users\Admin\Documents\1. UET\lab\VulGuardVN\GRACE-improve\baseline\baseline2\03_build_feature_store.py
I0000 00:00:1776678391.724342    5424 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776678396.336048    5424 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
[feature-store] dataset=devign split=train | rows=21854 | semantic_model=microsoft/unixcoder-base-nine | graph=auto
[feature-store] prepared graph view 256/21854 in 8.7s
[feature-store] prepared graph view 512/21854 in 11.1s
[feature-store] prepared graph view 768/21

In [6]:
run_script('04_train_hybrid_prefilter.py')


Running: c:\Users\Admin\Documents\miniconda\envs\nckh\python.exe C:\Users\Admin\Documents\1. UET\lab\VulGuardVN\GRACE-improve\baseline\baseline2\04_train_hybrid_prefilter.py
I0000 00:00:1776678770.662740    2660 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776678772.441466    2660 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776678774.556466    2660 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI AVX512_B

In [7]:
run_script('05_calibrate_budget_controller.py')


Running: c:\Users\Admin\Documents\miniconda\envs\nckh\python.exe C:\Users\Admin\Documents\1. UET\lab\VulGuardVN\GRACE-improve\baseline\baseline2\05_calibrate_budget_controller.py
I0000 00:00:1776679230.980518   10636 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776679232.853747   10636 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776679235.012714   10636 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI AVX

In [8]:
run_script('06_build_demo_bank.py')


Running: c:\Users\Admin\Documents\miniconda\envs\nckh\python.exe C:\Users\Admin\Documents\1. UET\lab\VulGuardVN\GRACE-improve\baseline\baseline2\06_build_demo_bank.py
Building demo bank for dataset=devign with 8000 sampled records (semantic=auto, graph=auto)
[demo-bank] Building graph features for 8000 records (graph=heuristic, semantic=auto)
[demo-bank] Graph backend notice: Joern health probe failed: Joern export returned no nodes.
[demo-bank] Graph features ready: 250/8000 in 0.2s
[demo-bank] Graph features ready: 500/8000 in 0.6s
[demo-bank] Graph features ready: 750/8000 in 0.9s
[demo-bank] Graph features ready: 1000/8000 in 1.3s
[demo-bank] Graph features ready: 1250/8000 in 1.6s
[demo-bank] Graph features ready: 1500/8000 in 1.9s
[demo-bank] Graph features ready: 1750/8000 in 2.2s
[demo-bank] Graph features ready: 2000/8000 in 2.5s
[demo-bank] Graph features ready: 2250/8000 in 2.8s
[demo-bank] Graph features ready: 2500/8000 in 3.1s
[demo-bank] Graph features ready: 2750/8000 i

In [ ]:
run_script('07_run_grace_hybrid.py')


Running: c:\Users\Admin\Documents\miniconda\envs\nckh\python.exe C:\Users\Admin\Documents\1. UET\lab\VulGuardVN\GRACE-improve\baseline\baseline2\07_run_grace_hybrid.py
I0000 00:00:1776769645.179271   10860 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776769651.382641   10860 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776769664.739780   10860 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 AVX512F AVX512_VNNI AVX512_BF16 FM

In [ ]:
run_script('08_evaluate_predictions.py')
